# TCM-FuzzyWiki V5.0 · MiniMax-M3 Colab（多并发 + 断点续跑）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/TCM-FuzzyWiki/blob/main/colab/TCM_FuzzyWiki_MiniMax_M3_Colab.ipynb)

完整复现 `https://github.com/pariskang/TCM-FuzzyWiki`，用 **MiniMax-M3** 做 observation-first 抽取，再由仓库内**确定性 pipeline** 复算 membership / 共现 / 规则 / Larsen 推理 / Mamdani / 三层聚合 / 关系网络 / Markdown Wiki / validation / audit / manifest。

- 支持 **OpenAI 协议**（`/v1`）与 **Anthropic 协议**（`/anthropic`）接入 MiniMax-M3，`PROVIDER` 一键切换。
- **多并发**抽取（`WORKERS`）。
- **chunk 级断点续跑**：Colab 断线 / 限流 / 崩溃后，重跑同一 cell 只补抽失败或缺失的 chunk。
- observation ID 按输入顺序确定性分配，**结果可复现**。

> 安全：**不要**把 API Key 写进 notebook。请用 Colab 左侧 🔑 Secrets 设置 `MINIMAX_API_KEY`，或运行时手动输入。


## 1. 挂载 Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库并安装


In [ ]:
%cd /content
!rm -rf TCM-FuzzyWiki
!git clone https://github.com/pariskang/TCM-FuzzyWiki.git
%cd /content/TCM-FuzzyWiki
# 该分支已含 build-llm / normalize-input；合并入 main 后此步可忽略
!git checkout claude/elegant-mccarthy-p7zg2g || echo "using default branch"
!git rev-parse --short HEAD

!python -m pip install -q -e '.[dev]'
# 可选 SDK 与 JSON 修复（Anthropic 协议装了 anthropic 会自动走 SDK）
!python -m pip install -q --upgrade openai anthropic json-repair

# 离线 demo + 单测，确认环境与确定性 pipeline 正常
!tcm-fuzzywiki run-demo --output /content/drive/MyDrive/TCM_FuzzyWiki_MiniMax_M3/_demo
!python -m pytest -q

## 3. 配置 API Key 与运行参数


In [ ]:
import os, getpass, datetime
from pathlib import Path

key = None
try:
    from google.colab import userdata
    key = userdata.get('MINIMAX_API_KEY')
except Exception:
    key = None
key = key or os.environ.get('MINIMAX_API_KEY')
if not key:
    key = getpass.getpass('请输入 MiniMax API Key（不显示）：')
os.environ['MINIMAX_API_KEY'] = key
print('API Key loaded:', bool(os.environ.get('MINIMAX_API_KEY')))

# ===== 可调参数 =====
PROVIDER = 'openai'        # 'openai' -> https://api.minimaxi.com/v1
                           # 'anthropic' -> https://api.minimaxi.com/anthropic
MODEL = 'MiniMax-M3'
WORKERS = 6                # 并发；遇到 429/限流就调小（如 2-3）
CHUNK_CHARS = 2400         # 长章节切块大小
MAX_OBS_PER_CHUNK = 12
THINKING = 'disabled'      # 结构化抽取建议关闭思考以稳定 JSON

# 输入古籍表格（任意列名，下一步会自动规范化）
RAW_INPUT = Path('/content/drive/MyDrive/中医古籍指令数据/book_chapters_split.xlsx')

# 绝对路径，避免 cell 间 cwd 变化导致找不到 config
CONFIG = '/content/TCM-FuzzyWiki/configs/tcm_fuzzywiki.yaml'

RUN_TAG = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive/TCM_FuzzyWiki_MiniMax_M3')
OUTPUT_DIR = DRIVE_ROOT / f'run_{RUN_TAG}'
SMOKE_DIR = OUTPUT_DIR / '_smoke'
NORMALIZED = OUTPUT_DIR / 'input' / 'chapters.normalized.xlsx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_INPUT.exists(), f'找不到输入文件：{RAW_INPUT}'
print('PROVIDER:', PROVIDER, '| OUTPUT_DIR:', OUTPUT_DIR)

## 4. 规范化任意表格为推荐字段

自动映射中/英文列名、猜测正文列、补默认 metadata，并输出可审计的列映射报告。


In [ ]:
!tcm-fuzzywiki normalize-input \
  --input "$RAW_INPUT" \
  --output "$NORMALIZED" \
  --report "$OUTPUT_DIR/input/column_mapping_report.json"

NORMALIZED_CSV = NORMALIZED.with_suffix('.csv')
import pandas as pd
print('规范化 CSV：', NORMALIZED_CSV)
display(pd.read_csv(NORMALIZED_CSV).head(3))

## 5. 配置/数据体检（doctor）


In [ ]:
!tcm-fuzzywiki doctor --config "$CONFIG" --input "$NORMALIZED_CSV" 

## 6. 小样本试跑（先验证 3 章，写入独立 _smoke 目录）


In [ ]:
!tcm-fuzzywiki build-llm \
  --input "$NORMALIZED_CSV" \
  --config "$CONFIG" \
  --output "$SMOKE_DIR" \
  --provider "$PROVIDER" \
  --model "$MODEL" \
  --workers 3 \
  --chunk-chars $CHUNK_CHARS \
  --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
  --thinking "$THINKING" \
  --limit 3

!cat "$SMOKE_DIR/summary.txt" 

## 7. 全量运行（多并发 + 断点续跑）

直接**重跑本 cell 即可续跑**：`success` 的 chunk 自动跳过，只补抽失败/缺失 chunk。
续跑时切勿修改 `CHUNK_CHARS` 或更换输入文件（manifest 会拒绝错配）。


In [ ]:
!tcm-fuzzywiki build-llm \
  --input "$NORMALIZED_CSV" \
  --config "$CONFIG" \
  --output "$OUTPUT_DIR" \
  --provider "$PROVIDER" \
  --model "$MODEL" \
  --workers $WORKERS \
  --chunk-chars $CHUNK_CHARS \
  --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
  --thinking "$THINKING"

!cat "$OUTPUT_DIR/summary.txt" 

## 8. 实时进度 / 断点状态

抽取中可随时运行查看；中断后用于确认断点。


In [ ]:
import pandas as pd
from pathlib import Path
ext = Path(OUTPUT_DIR) / 'extraction'
ls = ext / 'live_status.txt'
print('--- live_status.txt ---')
print(ls.read_text() if ls.exists() else '尚未开始')
sp = ext / 'source_progress.csv'
if sp.exists():
    df = pd.read_csv(sp)
    display(df['status'].value_counts(dropna=False))
    display(df[df['status'].isin(['error', 'partial_success'])].head(20))

## 9. 关键产物检查


In [ ]:
import pandas as pd, json
from pathlib import Path
data = Path(OUTPUT_DIR) / 'data'
for name in ['observations.csv','memberships.csv','candidate_patterns.csv','rules.csv',
             'inference_results.csv','aggregation_results.csv','relation_edges.csv',
             'validation_report.csv','completion_assessment.csv','run_manifest.json']:
    p = data / name
    print(('OK  ' if p.exists() else 'MISS'), name)

display(pd.read_csv(data / 'observations.csv').head(10))
display(pd.read_csv(data / 'completion_assessment.csv'))
man = json.loads((data / 'run_manifest.json').read_text())
print('provider:', man.get('llm_provider'), '| model:', man.get('llm_model'))
print('extraction_report:', json.dumps(man.get('extraction_report', {}), ensure_ascii=False))
print('Wiki 首页：', Path(OUTPUT_DIR) / 'wiki' / 'index.md')

## 10. 可选：金标准评估 / 专家规则

- 加 `--gold-dir path/to/gold` 计算 FCR/CRP/MIC/SMB/FIA（缺金标准时输出 `needs_gold_standard` 模板）。
- 审核 `data/expert_rule_review.csv` 后整理为 `rules.csv`，加 `--rules-csv path/to/rules.csv` 重跑升格正式规则。
- 切换 Anthropic 协议：把第 3 格 `PROVIDER` 改成 `'anthropic'` 重跑即可（base URL 自动指向 `/anthropic`）。

> 再次提醒：请勿把 API Key 写入 notebook 或提交到仓库。
